# Cross-Industry Accelerator — Configuration
### Universal Setup for All Industry Implementations

This notebook defines the **industry configuration** used by all downstream notebooks.
Run this notebook first, then `%run` it from other notebooks to import the config.

**Supported Industries:** Healthcare, Finance, Insurance, Retail, CPG, Construction, Oil & Gas, Media, Law Firms, Telecom

---

### How It Works
1. Set `INDUSTRY` to your target industry key
2. The notebook auto-discovers CSV files from the data folder
3. Files are classified as `dim_*`, `fact_*`, or `stream_*` based on naming convention
4. All downstream notebooks inherit this configuration

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 0: LOAD ZERO TRUST SECURITY UTILITIES
# ════════════════════════════════════════════════════════════════════════
%pip install nbformat
%run ./ZT_Security_Utils.ipynb

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 1: SELECT YOUR INDUSTRY (with ZT validation)
# ════════════════════════════════════════════════════════════════════════

INDUSTRY = "advertising"  # Change this to your target industry

# Supported values:
# "healthcare", "finance", "insurance", "retail", "cpg",
# "construction", "oil_and_gas", "media", "law_firms", "telecom",
# "advertising"

# ZT: Validate industry against supported list
INDUSTRY = validate_industry(INDUSTRY)
log_audit_event("INDUSTRY_SELECTED", INDUSTRY, "Validated against supported list")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2: FABRIC ARTIFACT NAMES
# ════════════════════════════════════════════════════════════════════════
# These follow the naming convention from INDUSTRY_REPLICATION_GUIDE.md
# Override any value below if your workspace uses different names.

INDUSTRY_DISPLAY_NAMES = {
    "healthcare":   "Healthcare",
    "finance":      "Finance",
    "insurance":    "Insurance",
    "retail":       "Retail",
    "cpg":          "CPG",
    "construction": "Construction",
    "oil_and_gas":  "OilGas",
    "media":        "Media",
    "law_firms":    "LawFirm",
    "telecom":      "Telecom",
    "advertising":  "Advertising"
}

INDUSTRY_LABEL = INDUSTRY_DISPLAY_NAMES.get(INDUSTRY, INDUSTRY.title())

# ── Fabric Artifact Names ──────────────────────────────────────────────
LAKEHOUSE_NAME     = f"{INDUSTRY_LABEL}_Data_Bronze"
WAREHOUSE_NAME     = f"{INDUSTRY_LABEL}_Datawarehouse"
EVENTHOUSE_NAME    = f"{INDUSTRY.replace(' ', '_')}_rt_store"
ONTOLOGY_NAME      = f"{INDUSTRY_LABEL}DocBurdenOntology"
DATA_AGENT_NAME    = f"{INDUSTRY_LABEL}QA"
OPS_AGENT_NAME     = f"{INDUSTRY_LABEL}DocBurden"
SEMANTIC_MODEL_NAME = f"{INDUSTRY_LABEL}_DocBurden_Model"

print(f"Industry:       {INDUSTRY} ({INDUSTRY_LABEL})")
print(f"Lakehouse:      {LAKEHOUSE_NAME}")
print(f"Warehouse:      {WAREHOUSE_NAME}")
print(f"Eventhouse:     {EVENTHOUSE_NAME}")
print(f"Ontology:       {ONTOLOGY_NAME}")
print(f"Data Agent:     {DATA_AGENT_NAME}")
print(f"Ops Agent:      {OPS_AGENT_NAME}")
print(f"Semantic Model: {SEMANTIC_MODEL_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2b: CREATE LAKEHOUSE IF IT DOESN'T EXIST
# ════════════════════════════════════════════════════════════════════════
# Works in both Fabric runtime (sempy) and local/VS Code (Azure CLI fallback)

import requests, json, time, subprocess

FABRIC_RUNTIME = False
try:
    import sempy.fabric as fabric
    FABRIC_RUNTIME = True
except ImportError:
    pass

if FABRIC_RUNTIME:
    # ── Running inside Fabric notebook ──
    workspace_id = fabric.get_workspace_id()
    items_df = fabric.list_items()
    lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]

    if not lh_matches.empty:
        print(f"✓ Lakehouse '{LAKEHOUSE_NAME}' already exists → {lh_matches.iloc[0].Id}")
    else:
        print(f"⚠ Lakehouse '{LAKEHOUSE_NAME}' not found. Creating...")
        access_token = notebookutils.credentials.getToken('pbi')
        headers = {
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        }
        body = {
            "displayName": LAKEHOUSE_NAME,
            "type": "Lakehouse",
        }
        resp = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items",
            headers=headers,
            json=body,
        )

        if resp.status_code in (200, 201, 202):
            if resp.status_code == 202:
                operation_url = resp.headers.get("Location", "")
                print(f"  ⏳ Provisioning (202)... polling for completion.")
                for _ in range(30):
                    time.sleep(5)
                    poll = requests.get(operation_url, headers=headers)
                    if poll.status_code == 200:
                        status = poll.json().get("status", "")
                        if status == "Succeeded":
                            print(f"  ✓ Provisioning complete.")
                            break
                        elif status == "Failed":
                            raise RuntimeError(f"Lakehouse creation failed: {poll.json()}")
                else:
                    print("  ⚠ Timed out waiting for provisioning — check workspace manually.")
            else:
                result = resp.json()
                print(f"✅ Lakehouse '{LAKEHOUSE_NAME}' created → {result.get('id', 'N/A')}")

            items_df = fabric.list_items()
            lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
            if not lh_matches.empty:
                print(f"✅ Lakehouse '{LAKEHOUSE_NAME}' confirmed → {lh_matches.iloc[0].Id}")
            else:
                print(f"⚠ Lakehouse created but not yet visible. Re-run this cell after a few seconds.")
        else:
            print(f"✗ Failed to create Lakehouse: {resp.status_code} — {resp.text}")
            raise RuntimeError(f"Lakehouse creation failed: {resp.status_code} {resp.text}")

else:
    # ── Running locally (VS Code) — use Azure CLI for Fabric REST API ──
    print("⚠ Not running in Fabric runtime (sempy unavailable). Using Azure CLI fallback.")
    try:
        token = subprocess.check_output(
            ["az", "account", "get-access-token", "--resource", "https://api.fabric.microsoft.com",
             "--query", "accessToken", "-o", "tsv"],
            text=True
        ).strip()
        headers = {"Authorization": f"Bearer {token}"}

        # List workspaces to find lakehouses
        ws_resp = requests.get("https://api.fabric.microsoft.com/v1/workspaces", headers=headers)
        ws_resp.raise_for_status()
        workspaces = ws_resp.json().get("value", [])

        found = False
        for ws in workspaces:
            ws_id = ws["id"]
            lh_resp = requests.get(
                f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/lakehouses",
                headers=headers
            )
            if lh_resp.status_code != 200:
                continue
            for lh in lh_resp.json().get("value", []):
                if lh["displayName"] == LAKEHOUSE_NAME:
                    print(f"✓ Lakehouse '{LAKEHOUSE_NAME}' found in workspace '{ws['displayName']}' → {lh['id']}")
                    found = True
                    break
            if found:
                break

        if not found:
            print(f"⚠ Lakehouse '{LAKEHOUSE_NAME}' not found in any workspace.")
            print("  Create it in the Fabric portal, or run this notebook inside Fabric to auto-create.")

    except FileNotFoundError:
        print("✗ Azure CLI ('az') not found. Install it or run this notebook in Fabric.")
    except subprocess.CalledProcessError:
        print("✗ Azure CLI auth failed. Run 'az login' first.")

log_audit_event("LAKEHOUSE_CHECK", LAKEHOUSE_NAME, "Ensured lakehouse exists")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 3: DATA PATHS & BINDING CONFIGURATION
# ════════════════════════════════════════════════════════════════════════

# Path where CSV files are uploaded in the Lakehouse Files area
CSV_BASE_PATH = f"/lakehouse/default/Files/{INDUSTRY}_data"

# Lakehouse schema (use 'dbo' for schema-enabled lakehouses)
LAKEHOUSE_SCHEMA = "dbo"

# Warehouse schema
WAREHOUSE_SCHEMA = "dbo"

# Eventhouse connection (fill these in from your Fabric workspace)
EVENTHOUSE_CLUSTER_URI = ""   # e.g. https://<name>.<region>.kusto.fabric.microsoft.com
EVENTHOUSE_DATABASE     = ""   # your Eventhouse DB name

# Ontology package path (if using .iq package)
ONTOLOGY_PACKAGE_PATH = f"/lakehouse/default/Files/{INDUSTRY}_ontology_package.iq"

print(f"CSV source:    {CSV_BASE_PATH}")
print(f"LH schema:     {LAKEHOUSE_SCHEMA}")
print(f"WH schema:     {WAREHOUSE_SCHEMA}")
print(f"Eventhouse:    {EVENTHOUSE_CLUSTER_URI or '(not configured)'}")
print(f"Ontology pkg:  {ONTOLOGY_PACKAGE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 4: AUTO-DISCOVER DATA SOURCES (with ZT validation)
# ════════════════════════════════════════════════════════════════════════
# Scans the CSV folder and classifies tables by naming convention:
#   dim_*    → Dimension tables  → Lakehouse + Warehouse
#   fact_*   → Fact tables       → Lakehouse + Warehouse (batch) or Eventhouse (events)
#   stream_* → Streaming tables  → Eventhouse only
#
# Works in both Fabric runtime (lakehouse mount) and local/VS Code (datasets/ fallback)

import os
import json

# ── Resolve CSV path: Fabric mount or local datasets/ folder ───────────
INDUSTRY_DATASET_FOLDERS = {
    "healthcare":   "healthcare_nursing_documentation",
    "finance":      "finance_banking_operations",
    "insurance":    "insurance_claims_operations",
    "retail":       "retail_store_operations",
    "cpg":          "retail_store_operations",
    "construction": "construction_site_operations",
    "oil_and_gas":  "oil_gas_field_operations",
    "media":        "media_content_operations",
    "law_firms":    "law_firm_operations",
    "telecom":      "telecom_network_operations",
    "advertising":  "advertising_campaign_operations",
}

if os.path.exists(CSV_BASE_PATH):
    _RESOLVED_CSV_PATH = CSV_BASE_PATH
    print(f"✓ Using Fabric lakehouse path: {CSV_BASE_PATH}")
else:
    # Local fallback: use datasets/<industry>/data/ relative to repo root
    _notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    _repo_root = os.path.dirname(_notebook_dir) if os.path.basename(_notebook_dir) == "cross_industry_notebooks" else _notebook_dir
    _dataset_folder = INDUSTRY_DATASET_FOLDERS.get(INDUSTRY, f"{INDUSTRY}_operations")
    _RESOLVED_CSV_PATH = os.path.join(_repo_root, "datasets", _dataset_folder, "data")
    print(f"⚠ Fabric path not available. Using local fallback: {_RESOLVED_CSV_PATH}")

def discover_data_sources(base_path):
    """Scan CSV directory and classify files by naming convention."""
    dim_tables = []
    fact_tables_batch = []    # fact tables for Lakehouse/Warehouse
    fact_tables_event = []    # fact tables for Eventhouse (event-level)
    stream_tables = []        # stream tables for Eventhouse only

    try:
        files = [f for f in os.listdir(base_path) if f.endswith('.csv')]
    except FileNotFoundError:
        print(f"ERROR: Path not found: {base_path}")
        print("Upload your CSV files to the Lakehouse Files area first.")
        return dim_tables, fact_tables_batch, fact_tables_event, stream_tables

    for f in sorted(files):
        table_name = f.replace('.csv', '')

        # ZT: Validate table name as safe identifier
        try:
            sanitize_identifier(table_name)
        except ValueError as e:
            print(f"  ⚠ ZT: Skipping file with unsafe name: {f!r} — {e}")
            log_audit_event("TABLE_REJECTED", table_name, str(e), "BLOCKED")
            continue

# Run discovery (uses resolved path — Fabric mount or local datasets/)
DIM_TABLES, FACT_TABLES_BATCH, FACT_TABLES_EVENT, STREAM_TABLES = discover_data_sources(_RESOLVED_CSV_PATH)
        elif table_name.startswith('stream_'):
            stream_tables.append(table_name)
        elif table_name.startswith('fact_'):
            # Heuristic: event-level facts contain time-series / clickstream data
            # These keywords indicate event tables → Eventhouse
            event_keywords = [
                '_events', '_interactions', '_clickstream', '_scans',
                '_alerts', '_location', '_handoff', '_administration',
                '_vital', '_assessments', '_escalation', '_transfers',
                '_integrity', '_inspections', '_rfi_', '_change_order',
                '_filing', '_contract', '_metadata', '_approval',
                '_delivery', '_regulatory', '_outage', '_rca',
                '_dispatch', '_nms_', '_dms_', '_mam_',
            ]
            if any(kw in table_name for kw in event_keywords):
                fact_tables_event.append(table_name)
            else:
                fact_tables_batch.append(table_name)
        else:
            # ZT: Unknown prefix — log and skip (least privilege)
            print(f"  ⚠ ZT: Skipping file with unrecognized prefix: {table_name}")
            log_audit_event("TABLE_REJECTED", table_name, "No recognized prefix (dim_/fact_/stream_)", "BLOCKED")
            continue

    return dim_tables, fact_tables_batch, fact_tables_event, stream_tables

# Run discovery
DIM_TABLES, FACT_TABLES_BATCH, FACT_TABLES_EVENT, STREAM_TABLES = discover_data_sources(CSV_BASE_PATH)

# ZT: Validate all discovered table names
DIM_TABLES = validate_table_names(DIM_TABLES)
FACT_TABLES_BATCH = validate_table_names(FACT_TABLES_BATCH)

FACT_TABLES_EVENT = validate_table_names(FACT_TABLES_EVENT)print(f"Total CSV files:  {len(DIM_TABLES) + len(FACT_TABLES_BATCH) + len(FACT_TABLES_EVENT) + len(STREAM_TABLES)}")

STREAM_TABLES = validate_table_names(STREAM_TABLES)print(f"Eventhouse target: {len(EVENTHOUSE_TABLES)} tables")

print(f"Warehouse target: {len(WAREHOUSE_TABLES)} tables")

log_audit_event("DATA_DISCOVERY", CSV_BASE_PATH,print(f"Lakehouse target: {len(LAKEHOUSE_TABLES)} tables")

    f"Found {len(DIM_TABLES)} dim, {len(FACT_TABLES_BATCH)} fact-batch, "print(f"\n{'─'*60}")

    f"{len(FACT_TABLES_EVENT)} fact-event, {len(STREAM_TABLES)} stream tables")for t in STREAM_TABLES: print(f"  • {t}")

print(f"\nStreaming tables — Eventhouse ({len(STREAM_TABLES)}):")

# Combined views for different targetsfor t in FACT_TABLES_EVENT: print(f"  • {t}")

LAKEHOUSE_TABLES = DIM_TABLES + FACT_TABLES_BATCH   # All dims + batch facts → Lakehouseprint(f"\nFact tables — Event/Eventhouse ({len(FACT_TABLES_EVENT)}):")

WAREHOUSE_TABLES = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT  # All non-streaming → Warehousefor t in FACT_TABLES_BATCH: print(f"  • {t}")

EVENTHOUSE_TABLES = FACT_TABLES_EVENT + STREAM_TABLES  # Events + streams → Eventhouseprint(f"\nFact tables — Batch/Lakehouse ({len(FACT_TABLES_BATCH)}):")

for t in DIM_TABLES: print(f"  • {t}")

print(f"\n{'='*60}")print(f"\nDimension tables ({len(DIM_TABLES)}):")

print(f"DATA SOURCE DISCOVERY — {INDUSTRY.upper()}")print(f"{'='*60}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 5: SCHEMA INSPECTION (preview any table)
# ════════════════════════════════════════════════════════════════════════
# Works in both Fabric runtime (spark) and local/VS Code (pandas fallback)

_USE_SPARK = False
try:
    spark  # noqa: F821 — defined in Fabric runtime
    _USE_SPARK = True
except NameError:
    import pandas as pd

def preview_table(table_name, base_path=_RESOLVED_CSV_PATH, rows=5):
    """Read a CSV and display schema + sample rows."""
    path = f"{base_path}/{table_name}.csv"

    if _USE_SPARK:
        df = spark.read.option("header", True).option("inferSchema", True).csv(path)
        print(f"\n{'─'*60}")
        print(f"Table: {table_name}  |  {df.count()} rows  |  {len(df.columns)} columns")

        print(f"{'─'*60}")    preview_table(FACT_TABLES_BATCH[0])

        df.printSchema()if FACT_TABLES_BATCH:

        df.show(rows, truncate=False)    preview_table(DIM_TABLES[0])

        return dfif DIM_TABLES:

    else:# Preview the first dimension and first fact table

        df = pd.read_csv(path)

        print(f"\n{'─'*60}")        return df

        print(f"Table: {table_name}  |  {len(df)} rows  |  {len(df.columns)} columns")        print(df.head(rows).to_string(index=False))

        print(f"{'─'*60}")        print(f"\nSample ({rows} rows):")

        print(f"\nSchema:")            print(f"  |-- {col}: {df[col].dtype}")
        for col in df.columns:

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG SUMMARY (exported to downstream notebooks via %run)
# ════════════════════════════════════════════════════════════════════════

CONFIG = {
    "industry":             INDUSTRY,
    "industry_label":       INDUSTRY_LABEL,
    "lakehouse_name":       LAKEHOUSE_NAME,
    "warehouse_name":       WAREHOUSE_NAME,
    "eventhouse_name":      EVENTHOUSE_NAME,
    "eventhouse_uri":       EVENTHOUSE_CLUSTER_URI,
    "eventhouse_db":        EVENTHOUSE_DATABASE,
    "ontology_name":        ONTOLOGY_NAME,
    "ontology_package":     ONTOLOGY_PACKAGE_PATH,
    "data_agent_name":      DATA_AGENT_NAME,
    "ops_agent_name":       OPS_AGENT_NAME,
    "semantic_model_name":  SEMANTIC_MODEL_NAME,
    "csv_base_path":        CSV_BASE_PATH,
    "lakehouse_schema":     LAKEHOUSE_SCHEMA,
    "warehouse_schema":     WAREHOUSE_SCHEMA,
    "dim_tables":           DIM_TABLES,
    "fact_tables_batch":    FACT_TABLES_BATCH,
    "fact_tables_event":    FACT_TABLES_EVENT,
    "stream_tables":        STREAM_TABLES,
    "lakehouse_tables":     LAKEHOUSE_TABLES,
    "warehouse_tables":     WAREHOUSE_TABLES,
    "eventhouse_tables":    EVENTHOUSE_TABLES,
}

print(json.dumps({k: v if not isinstance(v, list) else f"{len(v)} tables" for k, v in CONFIG.items()}, indent=2))